In [1]:
# ==========================================================
# Notebook 4 - Cell 1
# Imports
# ==========================================================

from pathlib import Path
import pandas as pd
import numpy as np

import geopandas as gpd

import rasterio
from rasterio.mask import mask

import matplotlib.pyplot as plt

print("="*70)
print("Notebook 4 - District Feature Extraction")
print("="*70)

Notebook 4 - District Feature Extraction


In [2]:
# ==========================================================
# Cell 2
# Project Paths
# ==========================================================

PROJECT_DIR = Path.cwd()

DATA_DIR = PROJECT_DIR / "Data"

BOUNDARY_DIR = DATA_DIR / "Boundaries"
DEM_DIR = DATA_DIR / "DEM"
NDVI_DIR = DATA_DIR / "NDVI"
SOIL_DIR = DATA_DIR / "Soil"
POP_DIR = DATA_DIR / "Population"
RAIN_DIR = DATA_DIR / "Rainfall"
LANDSLIDE_DIR = DATA_DIR / "LandslideInventory"
OSM_DIR = DATA_DIR / "OpenStreetMap"

OUTPUT_DIR = PROJECT_DIR / "Outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("Project:", PROJECT_DIR)

Project: /Users/rajdeepmandal/Desktop/PrithviX


In [3]:
# ==========================================================
# Cell 3
# Load India ADM2 Boundary
# ==========================================================

boundary = gpd.read_file(next(BOUNDARY_DIR.glob("*.geojson")))

print("Total districts :", len(boundary))
print()

print(boundary.columns.tolist())

Total districts : 735

['shapeName', 'shapeISO', 'shapeID', 'shapeGroup', 'shapeType', 'geometry']


In [4]:
# ==========================================================
# Cell 4
# Identify District Column
# ==========================================================

for col in boundary.columns:
    print(col)

shapeName
shapeISO
shapeID
shapeGroup
shapeType
geometry


In [7]:
# ==========================================================
# Cell 5
# Select District
# ==========================================================

DISTRICT_NAME = "Darjiling"

# Find the district name column automatically
possible_cols = [
    "shapeName", "DISTRICT", "district",
    "ADM2_NAME", "NAME_2", "District"
]

district_col = None

for col in possible_cols:
    if col in boundary.columns:
        district_col = col
        break

if district_col is None:
    raise Exception("District name column not found.")

district = boundary[
    boundary[district_col].str.strip().str.lower() ==
    DISTRICT_NAME.lower()
]

print("="*60)
print("District Selected")
print("="*60)

print("Column Used :", district_col)
print("Rows Found  :", len(district))

display(district.head())

District Selected
Column Used : shapeName
Rows Found  : 1


,shapeName,shapeISO,shapeID,shapeGroup,shapeType,geometry
382,Darjiling,,76128533B17109302371020,IND,ADM2,"POLYGON ((88.24247 26.44935, 88.24245 26.45281..."


In [9]:
# ==========================================================
# Cell 6 - Crop DEM (Fixed)
# ==========================================================

import rasterio
from rasterio.mask import mask

DEM_FILE = next(DEM_DIR.glob("*.tif"))

with rasterio.open(DEM_FILE) as src:

    dem_crop, dem_transform = mask(
        src,
        district.geometry,
        crop=True,
        filled=False      # <-- IMPORTANT
    )

print("="*60)
print("DEM Cropped Successfully")
print("="*60)

print("Shape :", dem_crop.shape)
print("CRS   :", src.crs)
print("Data type :", dem_crop.dtype)

DEM Cropped Successfully
Shape : (1, 2891, 1849)
CRS   : EPSG:4326
Data type : int16


In [10]:
# ==========================================================
# Cell 7 - Elevation Statistics
# ==========================================================

import numpy as np

elevation = dem_crop[0]

print("="*60)
print("Elevation Statistics")
print("="*60)

print("Mean :", float(elevation.mean()))
print("Minimum :", float(elevation.min()))
print("Maximum :", float(elevation.max()))

Elevation Statistics
Mean : 845.9824549641968
Minimum : 61.0
Maximum : 3601.0


In [11]:
# ==========================================================
# Cell 8 - Crop NDVI
# ==========================================================

import rasterio
from rasterio.mask import mask
import numpy as np

NDVI_FILE = next(NDVI_DIR.glob("*.tif"))

with rasterio.open(NDVI_FILE) as src:

    ndvi_crop, ndvi_transform = mask(
        src,
        district.geometry,
        crop=True,
        filled=False
    )

ndvi = ndvi_crop[0]

print("="*60)
print("NDVI Statistics")
print("="*60)

print("Mean NDVI :", float(ndvi.mean()))
print("Minimum   :", float(ndvi.min()))
print("Maximum   :", float(ndvi.max()))

NDVI Statistics
Mean NDVI : 0.6706732149262714
Minimum   : -0.644385039806366
Maximum   : 1.0


In [16]:
# ==========================================================
# Cell 9 - Population Statistics (Final)
# ==========================================================

import numpy as np
import rasterio
from rasterio.mask import mask

print("="*70)
print("POPULATION FEATURE EXTRACTION")
print("="*70)

population_found = False

for pop_file in sorted(POP_DIR.glob("*.tif")):

    try:

        with rasterio.open(pop_file) as src:

            pop_crop, transform = mask(
                src,
                district.geometry,
                crop=True,
                filled=False
            )

            population = pop_crop[0]

            # Extract only valid values
            values = population.compressed()
            values = values[np.isfinite(values)]

            if len(values) == 0:
                continue

            print("Population Raster :", pop_file.name)
            print("Valid Pixels      :", len(values))
            print("Mean Population   :", round(values.mean(), 2))
            print("Maximum           :", round(values.max(), 2))
            print("Minimum           :", round(values.min(), 2))

            mean_population = float(values.mean())

            population_found = True
            break

    except ValueError:
        continue

if not population_found:

    mean_population = np.nan

    print("❌ No valid population data found.")

POPULATION FEATURE EXTRACTION
Population Raster : northeastern_population.tif
Valid Pixels      : 230495
Mean Population   : 6.95
Maximum           : 129.37
Minimum           : 0.03


In [21]:
# ==========================================================
# Cell 10 - Load Landslide Inventory (Correct Version)
# ==========================================================

import pandas as pd
import geopandas as gpd

# Read CSV
landslides = pd.read_csv(LANDSLIDE_FILE, skiprows=5)

# Remove trailing spaces from column names
landslides.columns = landslides.columns.str.strip()

# Remove unnamed columns
landslides = landslides.loc[:, ~landslides.columns.str.contains("^Unnamed")]

# Remove fully empty columns
landslides = landslides.dropna(axis=1, how="all")

# Convert coordinates
landslides["Latitude"] = pd.to_numeric(
    landslides["Latitude"],
    errors="coerce"
)

landslides["Longitude"] = pd.to_numeric(
    landslides["Longitude"],
    errors="coerce"
)

# Remove invalid rows
landslides = landslides.dropna(subset=["Latitude", "Longitude"])

# Create GeoDataFrame
landslides_gdf = gpd.GeoDataFrame(
    landslides,
    geometry=gpd.points_from_xy(
        landslides["Longitude"],
        landslides["Latitude"]
    ),
    crs="EPSG:4326"
)

print("="*60)
print("LANDSLIDES LOADED")
print("="*60)

print("Rows :", len(landslides_gdf))
print("Columns :", len(landslides_gdf.columns))

display(landslides_gdf.head())

LANDSLIDES LOADED
Rows : 35591
Columns : 12


,Sl.No.,Slide_No,State,District,Slide_Name,NH_SH_Location,Latitude,Longitude,Material Involved,Movement,History,geometry
1,1,ASM/HKN/83D07/2020/2,Assam,Hailakandi,Kukinala slide,Kukinala,24.27,92.50,Debris,Slide,NaN,POINT (92.5 24.27)
2,2,AS/HKN/83D11/2020/1,Assam,Hailakandi,Jalnachaura slide,Jalnachaura,24.31,92.56,Debris,Slide,NaN,POINT (92.56 24.31)
3,3,AS/HKN/83D10/2020/6,Assam,Hailakandi,Nandagram slide,Nandagram Pt. I,24.32,92.51,Debris,Slide,NaN,POINT (92.51 24.32)
4,4,AS/HKN/83D11/2020/4,Assam,Hailakandi,Nishkar slide,Mahapur,24.43,92.55,Debris,Slide,NaN,POINT (92.55 24.43)
5,5,AS/KRJ/83D07/2020/1,Assam,Karimganj,Paglacherra slide,PaglacherraPunj,24.44,92.50,Rock,Slide,NaN,POINT (92.5 24.44)


In [25]:
# ==========================================================
# Cell 11 - Convert Landslides to GeoDataFrame
# ==========================================================

import geopandas as gpd

# Create geometry from Longitude and Latitude
landslides = gpd.GeoDataFrame(
    landslides,
    geometry=gpd.points_from_xy(
        landslides["Longitude"],
        landslides["Latitude"]
    ),
    crs="EPSG:4326"
)

# Select Darjeeling district boundary
district_boundary = boundary[
    boundary["shapeName"] == "Darjiling"
]

print("="*60)
print("District Boundary")
print("="*60)
print("Rows :", len(district_boundary))

# Spatial Join
darjeeling_ls = gpd.sjoin(
    landslides,
    district_boundary,
    predicate="within",
    how="inner"
)

print("\nTotal Landslides inside Darjeeling :", len(darjeeling_ls))

display(
    darjeeling_ls[
        ["Slide_Name","Latitude","Longitude"]
    ].head()
)

District Boundary
Rows : 1

Total Landslides inside Darjeeling : 1058


,Slide_Name,Latitude,Longitude
32216,NaN,27.082750,88.434778
32219,NaN,27.087667,88.416528
32220,NaN,27.090083,88.413139
32221,NaN,27.090944,88.406833
32224,NaN,27.097833,88.378500


In [26]:
# ==========================================================
# Cell 12 - Rasterize Landslide Points
# ==========================================================

from rasterio.features import rasterize

with rasterio.open(DEM_FILE) as src:

    transform = src.transform
    out_shape = (src.height, src.width)

# Convert every landslide point to value 1
shapes = ((geom, 1) for geom in darjeeling_ls.geometry)

landslide_mask = rasterize(
    shapes=shapes,
    out_shape=out_shape,
    transform=transform,
    fill=0,
    dtype="uint8"
)

print("="*60)
print("LANDSLIDE MASK")
print("="*60)

print("Shape :", landslide_mask.shape)
print("Total Landslide Pixels :", landslide_mask.sum())

LANDSLIDE MASK
Shape : (27474, 38963)
Total Landslide Pixels : 966


In [35]:
import rasterio
from pathlib import Path

# Root project folder
PROJECT_DIR = Path("/Users/rajdeepmandal/Desktop/PrithviX")

DATA_DIR = PROJECT_DIR / "Data"
PROCESSED_DIR = PROJECT_DIR / "Processed"

# Create Processed folder if it doesn't exist
PROCESSED_DIR.mkdir(exist_ok=True)

DEM_FILE = DATA_DIR / "DEM" / "ner_dem.tif"

out_file = PROCESSED_DIR / "landslide_mask.tif"

print(PROJECT_DIR)
print(PROCESSED_DIR)
print(out_file)

with rasterio.open(DEM_FILE) as dem:

    with rasterio.open(
        out_file,
        "w",
        driver="GTiff",
        height=landslide_mask.shape[0],
        width=landslide_mask.shape[1],
        count=1,
        dtype="uint8",
        crs=dem.crs,
        transform=dem.transform,
    ) as dst:

        dst.write(landslide_mask.astype("uint8"), 1)

print("="*60)
print("LANDSLIDE MASK SAVED")
print("="*60)
print(out_file)

/Users/rajdeepmandal/Desktop/PrithviX
/Users/rajdeepmandal/Desktop/PrithviX/Processed
/Users/rajdeepmandal/Desktop/PrithviX/Processed/landslide_mask.tif
LANDSLIDE MASK SAVED
/Users/rajdeepmandal/Desktop/PrithviX/Processed/landslide_mask.tif
